# Frequency-Severity Model Fitting

See [`reports/wildfire_loss_report.md`](../reports/wildfire_loss_report.md) for full narrative, interpretation, and limitations. Region config: `src/common.py`.

In [ ]:
%matplotlib inline
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tools.sm_exceptions import ConvergenceWarning, HessianInversionWarning

from common import REGION_NAME, PROCESSED_DIR

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

DATA_PROCESSED = "../data/processed"
print(f"Region: {REGION_NAME}")


Region: alberta


## Model evolution

| Step | Motivation |
|---|---|
| Poisson | Initial baseline |
| NB | Severe overdispersion |
| Feature reduction | Reduce multicollinearity |
| Fuel feature | Add geospatial information |
| LOEO | Evaluate generalization |
| GEE | Explore within-event dependence |
| SSP projection | Illustrative future scenario |

In [2]:
steps = [
    "Poisson",
    "Overdispersion discovered",
    "Negative Binomial",
    "Multicollinearity discovered",
    "Feature reduction",
    "Fuel feature added",
    "Validation",
    "Spatial dependence",
    "Final model",
]
discovery_steps = {"Overdispersion discovered", "Multicollinearity discovered"}

fig, ax = plt.subplots(figsize=(4.5, 11))
n = len(steps)
for i, step in enumerate(steps):
    y = n - i
    is_discovery = step in discovery_steps
    is_final = step == "Final model"
    color = "#f2b134" if is_discovery else ("#55a868" if is_final else "#4c72b0")
    ax.add_patch(plt.Rectangle((0, y - 0.4), 1, 0.8, facecolor=color, edgecolor="none"))
    ax.text(0.5, y, step, ha="center", va="center", color="black" if is_discovery else "white",
            fontsize=10, fontweight="bold" if is_final else "normal")
    if y > 1:
        # arrow points DOWN: head (xy) at the bottom, tail (xytext) at the top
        ax.annotate("", xy=(0.5, y - 0.58), xytext=(0.5, y - 0.42),
                     arrowprops=dict(arrowstyle="-|>", color="gray", lw=1.5))

ax.set_xlim(-0.1, 1.1)
ax.set_ylim(0.3, n + 0.7)
ax.axis("off")
ax.set_title("Model evolution", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"../outputs/figures/{REGION_NAME}_model_evolution.png", dpi=150, bbox_inches="tight")
plt.show()


## 1. Load and prepare modelling data

In [3]:
claims_features = pd.read_csv(f"{DATA_PROCESSED}/features/{REGION_NAME}_claims_features.csv")
print("Full table:", claims_features.shape)

# Rows with no valid FWI signal (see EDA notebook) can't be used in an
# FWI-based model -- excluded, not imputed, and the exclusion is quantified
# rather than silently dropped.
excluded = claims_features[claims_features["climate_days_available"] == 0]
df = claims_features[claims_features["climate_days_available"] > 0].copy()

print(f"Excluded (zero valid FWI days): {len(excluded)} rows, "
      f"{excluded['Total $ Loss'].sum():,.0f} total $ loss "
      f"({excluded['Total $ Loss'].sum() / claims_features['Total $ Loss'].sum():.1%} of all loss)")
print(f"Modelling set: {len(df)} rows")


Full table: (305, 25)
Excluded (zero valid FWI days): 73 rows, 1,186,872 total $ loss (1.1% of all loss)
Modelling set: 232 rows


In [4]:
df["claim_count"] = (df["Loss Frequency"] * df["Number of Exposure_at_claim"]).round().astype(int)
df["severity"] = np.where(df["claim_count"] > 0, df["Total $ Loss"] / df["claim_count"], np.nan)

print(df[["claim_count", "severity"]].describe())
print()
print(f"Rows with claim_count > 0 (used for severity model): {(df['claim_count'] > 0).sum()}")


        claim_count      severity
count    232.000000     79.000000
mean     179.892241   1306.451378
std     1399.488825   2389.539848
min        0.000000      0.000000
25%        0.000000    140.413323
50%        0.000000    330.088776
75%       20.000000   1167.004240
max    15364.000000  15029.519082

Rows with claim_count > 0 (used for severity model): 79


## 2. Feature selection

In [5]:
PREDICTORS = ["fwi_max", "dc_max", "dominant_fuel_pct"]  # 2 climate + 1 geospatial; single source used below

print("Correlation between candidate predictors:")
print(df[PREDICTORS].corr())


Correlation between candidate predictors:
                    fwi_max    dc_max  dominant_fuel_pct
fwi_max            1.000000  0.353581           0.053094
dc_max             0.353581  1.000000           0.014586
dominant_fuel_pct  0.053094  0.014586           1.000000


## 3. Frequency model

In [6]:
X = sm.add_constant(df[PREDICTORS])
offset = np.log(df["Number of Exposure_at_claim"])

poisson_model = sm.GLM(df["claim_count"], X, family=sm.families.Poisson(), offset=offset).fit()
print(poisson_model.summary())

pearson_chi2 = poisson_model.pearson_chi2
dispersion = pearson_chi2 / poisson_model.df_resid
print(f"\nPearson chi2 / df = {dispersion:.1f} (>>1 indicates overdispersion -- Poisson SEs are then too small)")


                 Generalized Linear Model Regression Results                  
Dep. Variable:            claim_count   No. Observations:                  232
Model:                            GLM   Df Residuals:                      228
Model Family:                 Poisson   Df Model:                            3
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -67995.
Date:                Tue, 28 Jul 2026   Deviance:                   1.3554e+05
Time:                        16:28:14   Pearson chi2:                 2.59e+05
No. Iterations:                     8   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -0.1857      0.01

In [7]:
nb_model = sm.NegativeBinomial(df["claim_count"], X, offset=offset).fit(disp=0)
print(nb_model.summary())
print(f"\nAIC -- Poisson: {poisson_model.aic:.1f}, Negative Binomial: {nb_model.aic:.1f} (lower is better)")


                     NegativeBinomial Regression Results                      
Dep. Variable:            claim_count   No. Observations:                  232
Model:               NegativeBinomial   Df Residuals:                      228
Method:                           MLE   Df Model:                            3
Date:                Tue, 28 Jul 2026   Pseudo R-squ.:                 0.05955
Time:                        16:28:14   Log-Likelihood:                -593.42
converged:                       True   LL-Null:                       -631.00
Covariance Type:            nonrobust   LLR p-value:                 3.360e-16
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -0.5370      0.744     -0.722      0.470      -1.995       0.921
fwi_max               0.2387      0.178      1.339      0.181      -0.111       0.588
dc_max               -0.

In [8]:
from scipy import stats

# Whether dominant_fuel_pct earns its place should rest on whether it improves
# the model, not on whether its own p-value clears 0.05 -- with 232 rows and
# only 2 underlying events, p-values are easy to over-read. Nested-model
# comparison (likelihood ratio test + AIC) is the more defensible evidence.
reduced_predictors = ["fwi_max", "dc_max"]
X_reduced = sm.add_constant(df[reduced_predictors])
nb_model_no_fuel = sm.NegativeBinomial(df["claim_count"], X_reduced, offset=offset).fit(disp=0)

lr_stat = 2 * (nb_model.llf - nb_model_no_fuel.llf)
lr_df = len(PREDICTORS) - len(reduced_predictors)
lr_pvalue = stats.chi2.sf(lr_stat, lr_df)

print(f"Without dominant_fuel_pct -- AIC: {nb_model_no_fuel.aic:.1f}, log-likelihood: {nb_model_no_fuel.llf:.1f}")
print(f"With dominant_fuel_pct    -- AIC: {nb_model.aic:.1f}, log-likelihood: {nb_model.llf:.1f}")
print(f"AIC improvement from adding dominant_fuel_pct: {nb_model_no_fuel.aic - nb_model.aic:.1f} (lower AIC is better)")
print(f"Likelihood ratio test: LR = {lr_stat:.1f}, df = {lr_df}, p = {lr_pvalue:.2e}")


Without dominant_fuel_pct -- AIC: 1217.8, log-likelihood: -604.9
With dominant_fuel_pct    -- AIC: 1196.8, log-likelihood: -593.4
AIC improvement from adding dominant_fuel_pct: 21.0 (lower AIC is better)
Likelihood ratio test: LR = 23.0, df = 1, p = 1.65e-06


The Negative Binomial was selected because the Poisson assumption of equal mean and variance was strongly violated (Pearson χ²/df >> 1), resulting in a substantially lower AIC.

## 4. Severity model

In [9]:
zero_dollar_claims = df[(df["claim_count"] > 0) & (df["Total $ Loss"] == 0)]
print(f"Rows with claims filed but $0 total loss: {len(zero_dollar_claims)} -- {sorted(zero_dollar_claims['FSA'])}")
print("Real, not an error -- but Gamma requires severity > 0, so these are excluded from the severity fit only")
print("(they stay correctly included in the frequency model, which only needs claim_count).")

severity_df = df[(df["claim_count"] > 0) & (df["Total $ Loss"] > 0)].copy()
X_sev = sm.add_constant(severity_df[PREDICTORS])

severity_model = sm.GLM(severity_df["severity"], X_sev, family=sm.families.Gamma(link=sm.families.links.Log())).fit()
print(severity_model.summary())


Rows with claims filed but $0 total loss: 3 -- ['T6C', 'T6E', 'T7Y']
Real, not an error -- but Gamma requires severity > 0, so these are excluded from the severity fit only
(they stay correctly included in the frequency model, which only needs claim_count).
                 Generalized Linear Model Regression Results                  
Dep. Variable:               severity   No. Observations:                   76
Model:                            GLM   Df Residuals:                       72
Model Family:                   Gamma   Df Model:                            3
Link Function:                    Log   Scale:                          2.7298
Method:                          IRLS   Log-Likelihood:                -615.39
Date:                Tue, 28 Jul 2026   Deviance:                       159.60
Time:                        16:28:14   Pearson chi2:                     197.
No. Iterations:                    60   Pseudo R-squ. (CS):            0.05187
Covariance Type:            non

## 5. Combined expected loss - in sample check

In [10]:
best_freq_model = nb_model if nb_model.aic < poisson_model.aic else poisson_model
print(f"Using {'Negative Binomial' if best_freq_model is nb_model else 'Poisson'} for frequency.")

df["predicted_count"] = best_freq_model.predict(X, offset=offset)
df["predicted_severity"] = severity_model.predict(sm.add_constant(df[PREDICTORS], has_constant="add"))
df["predicted_loss"] = df["predicted_count"] * df["predicted_severity"]

fig, ax = plt.subplots(figsize=(6, 6))
sns.scatterplot(data=df, x="Total $ Loss", y="predicted_loss", hue="Loss Dates", alpha=0.6, ax=ax)
lims = [0, max(df["Total $ Loss"].max(), df["predicted_loss"].max())]
ax.plot(lims, lims, "k--", alpha=0.4, label="perfect fit")
ax.set_xscale("symlog")
ax.set_yscale("symlog")
ax.set_title("Actual vs. predicted Total $ Loss (in-sample)")
ax.legend()
plt.tight_layout()
plt.savefig(f"../outputs/figures/{REGION_NAME}_actual_vs_predicted_loss.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Actual total loss (modelling set): {df['Total $ Loss'].sum():,.0f}")
print(f"Predicted total loss (modelling set): {df['predicted_loss'].sum():,.0f}")


Using Negative Binomial for frequency.


Actual total loss (modelling set): 106,932,747
Predicted total loss (modelling set): 41,382,388


The aggregate total is in a plausible range, but row-level predictions vary by several orders of magnitude for FSAs with the same true (zero) loss. This indicates the model captures broad portfolio-level scale but not fine-grained, FSA-level risk differentiation.

In [11]:
# Numerical summary of in-sample fit, not just the scatterplot -- a small
# metrics table plus a breakdown by event, since a single pooled total can
# hide one event fitting much better than the other.
from IPython.display import display

resid = df["Total $ Loss"] - df["predicted_loss"]
fit_metrics = pd.DataFrame({
    "value": [resid.abs().mean(), np.sqrt((resid ** 2).mean()), resid.mean()],
}, index=["MAE", "RMSE", "Mean residual"])
fit_metrics["value (formatted)"] = fit_metrics["value"].map(lambda v: f"{v:,.0f}")
display(fit_metrics)

by_event = df.groupby("Loss Dates").agg(
    observed_total=("Total $ Loss", "sum"),
    predicted_total=("predicted_loss", "sum"),
)
by_event["predicted_pct_of_observed"] = (by_event["predicted_total"] / by_event["observed_total"] * 100).round(1)
by_event


,value,value (formatted)
MAE,5.727254e+05,"572,725"
RMSE,3.336833e+06,"3,336,833"
Mean residual,2.825447e+05,"282,545"


,observed_total,predicted_total,predicted_pct_of_observed
Loss Dates,,,
"July 22 - August 17, 2024",2.374869e+07,7.729895e+06,32.5
"May 3 - May 18, 2016",8.318406e+07,3.365249e+07,40.5


## 6. Model evaluation figures

In [12]:
aic_table = pd.DataFrame({
    "model": ["Poisson", "Negative Binomial"],
    "AIC": [poisson_model.aic, nb_model.aic],
}).set_index("model")
aic_table["AIC (formatted)"] = aic_table["AIC"].map(lambda v: f"{v:,.1f}")
aic_table.to_csv(f"../outputs/tables/model_comparison.csv")
aic_table


,AIC,AIC (formatted)
model,,
Poisson,135998.103073,"135,998.1"
Negative Binomial,1196.846463,"1,196.8"


In [13]:
def coef_table(model, label):
    t = pd.DataFrame({"coef": model.params, "std_err": model.bse, "p_value": model.pvalues})
    t.insert(0, "model", label)
    t.index.name = "term"
    return t.reset_index()

coef_summary = pd.concat([
    coef_table(nb_model, "Frequency (Negative Binomial)"),
    coef_table(severity_model, "Severity (Gamma)"),
], ignore_index=True)
coef_summary["significant (p<0.05)"] = coef_summary["p_value"] < 0.05
coef_summary.to_csv("../outputs/tables/coefficient_table.csv", index=False)
coef_summary


,term,model,coef,std_err,p_value,significant (p<0.05)
0,const,Frequency (Negative Binomial),-0.536997,0.743653,4.702292e-01,False
1,fwi_max,Frequency (Negative Binomial),0.238731,0.178310,1.806179e-01,False
2,dc_max,Frequency (Negative Binomial),-0.025198,0.003446,2.618321e-13,True
3,dominant_fuel_pct,Frequency (Negative Binomial),-4.457999,0.908304,9.198853e-07,True
4,alpha,Frequency (Negative Binomial),10.502437,1.371442,1.889179e-14,True
5,const,Severity (Gamma),7.843359,0.678768,6.941284e-31,True
6,fwi_max,Severity (Gamma),0.030259,0.164565,8.541110e-01,False
7,dc_max,Severity (Gamma),0.002549,0.003390,4.520155e-01,False
8,dominant_fuel_pct,Severity (Gamma),-1.455291,0.833141,8.068025e-02,False


In [14]:
# Visualizes coefficient magnitude and uncertainty.
plot_coefs = coef_summary[coef_summary["term"] != "alpha"].copy()
plot_coefs["ci_low"] = plot_coefs["coef"] - 1.96 * plot_coefs["std_err"]
plot_coefs["ci_high"] = plot_coefs["coef"] + 1.96 * plot_coefs["std_err"]

fig, ax = plt.subplots(figsize=(7, 5))
models = plot_coefs["model"].unique()
colors = {models[0]: "#4c72b0", models[1]: "#c44e52"}
y_labels, y_pos = [], []
y = 0
for term in plot_coefs["term"].unique():
    for model in models:
        row = plot_coefs[(plot_coefs["term"] == term) & (plot_coefs["model"] == model)]
        if row.empty:
            continue
        row = row.iloc[0]
        ax.errorbar(row["coef"], y, xerr=1.96 * row["std_err"], fmt="o", color=colors[model], capsize=3)
        y_labels.append(f"{term} ({model.split(' ')[0]})")
        y_pos.append(y)
        y += 1
    y += 0.5

ax.axvline(0, color="k", linestyle="--", alpha=0.4)
ax.set_yticks(y_pos)
ax.set_yticklabels(y_labels, fontsize=8)
ax.set_xlabel("Coefficient (95% CI)")
ax.set_title("Coefficient plot: frequency (NB) and severity (Gamma) models")
plt.tight_layout()
plt.savefig(f"../outputs/figures/{REGION_NAME}_coefficient_plot.png", dpi=150, bbox_inches="tight")
plt.show()


`dominant_fuel_pct` and `dc_max` sit clearly away from zero in the frequency model; `fwi_max` moves much closer to zero once fuel is included, and no severity coefficient is far from zero. This is consistent with the AIC/likelihood-ratio evidence above: the geospatial feature, not climate alone, is doing most of the work.

In [15]:
df["residual"] = df["Total $ Loss"] - df["predicted_loss"]

fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(data=df, x="predicted_loss", y="residual", hue="Loss Dates", alpha=0.6, ax=ax)
ax.axhline(0, color="k", linestyle="--", alpha=0.5)
ax.set_xscale("symlog")
ax.set_yscale("symlog")
ax.set_title("Residuals (actual - predicted) vs. predicted loss (in-sample)")
plt.tight_layout()
plt.savefig(f"../outputs/figures/{REGION_NAME}_residuals.png", dpi=150, bbox_inches="tight")
plt.show()


Residuals remain widely dispersed, indicating substantial unexplained variability. No strong systematic trend is visible, suggesting the model captures the overall relationship but not all local variation.

The Negative Binomial substantially improved model fit relative to the Poisson model. Residual diagnostics suggest the baseline captures broad trends but leaves considerable unexplained variation, consistent with the limited number of historical wildfire events.

## 7. Out-of-sample validation (leave-one-event-out)

Because only two independent wildfire events are available, conventional train/test splitting is inappropriate. Leave-one-event-out validation provides the strongest available assessment of model generalization.

In [16]:
def fit_and_evaluate(train, test, predictors, label):
    X_train = sm.add_constant(train[predictors])
    offset_train = np.log(train["Number of Exposure_at_claim"])
    freq_model = sm.NegativeBinomial(train["claim_count"], X_train, offset=offset_train).fit(disp=0)
    converged = freq_model.mle_retvals.get("converged", False)
    if not converged:
        print(f"  [{label}] WARNING: frequency model did not converge on this subsample -- results below are unreliable")

    sev_train = train[(train["claim_count"] > 0) & (train["Total $ Loss"] > 0)]
    X_sev_train = sm.add_constant(sev_train[predictors])
    sev_model = sm.GLM(sev_train["severity"], X_sev_train, family=sm.families.Gamma(link=sm.families.links.Log())).fit()

    X_test = sm.add_constant(test[predictors], has_constant="add")
    offset_test = np.log(test["Number of Exposure_at_claim"])
    predicted_count = freq_model.predict(X_test, offset=offset_test)
    predicted_severity = sev_model.predict(X_test)
    predicted_loss = predicted_count * predicted_severity

    results = pd.DataFrame({
        "FSA": test["FSA"].values,
        "actual": test["Total $ Loss"].values,
        "predicted": predicted_loss.values,
    })
    return results, converged


event_a = df[df["Loss Dates"] == "May 3 - May 18, 2016"]
event_b = df[df["Loss Dates"] == "July 22 - August 17, 2024"]

results_a_to_b, converged_a_to_b = fit_and_evaluate(event_a, event_b, PREDICTORS, "train=May, test=Jul/Aug")
results_b_to_a, converged_b_to_a = fit_and_evaluate(event_b, event_a, PREDICTORS, "train=Jul/Aug, test=May")

for name, res in [
    ("Trained on May 2016, tested on Jul/Aug 2024", results_a_to_b),
    ("Trained on Jul/Aug 2024, tested on May 2016", results_b_to_a),
]:
    print(f"{name}:")
    print(f"  actual total: {res['actual'].sum():,.0f}, predicted total: {res['predicted'].sum():,.0f}")
    print(f"  Spearman rank correlation (actual, predicted): {res['actual'].corr(res['predicted'], method='spearman'):.3f}")
    print()


  [train=May, test=Jul/Aug] WARNING: frequency model did not converge on this subsample -- results below are unreliable
  [train=Jul/Aug, test=May] WARNING: frequency model did not converge on this subsample -- results below are unreliable
Trained on May 2016, tested on Jul/Aug 2024:
  actual total: 23,748,687, predicted total: 186,362
  Spearman rank correlation (actual, predicted): 0.068

Trained on Jul/Aug 2024, tested on May 2016:
  actual total: 83,184,060, predicted total: 189,900,548
  Spearman rank correlation (actual, predicted): 0.297



In [17]:
def loeo_errors(res):
    err = res["actual"] - res["predicted"]
    return err.abs().mean(), np.sqrt((err ** 2).mean())

mae_a, rmse_a = loeo_errors(results_a_to_b)
mae_b, rmse_b = loeo_errors(results_b_to_a)

loeo_summary = pd.DataFrame({
    "training_event": ["May 2016", "Jul/Aug 2024"],
    "testing_event": ["Jul/Aug 2024", "May 2016"],
    "converged": [converged_a_to_b, converged_b_to_a],
    "MAE": [mae_a, mae_b],
    "RMSE": [rmse_a, rmse_b],
}).set_index("training_event")
loeo_summary


,testing_event,converged,MAE,RMSE
training_event,,,,
May 2016,Jul/Aug 2024,False,1.571725e+05,1.897999e+06
Jul/Aug 2024,May 2016,False,3.179053e+06,1.420665e+07


## 8. Spatial dependence

In [18]:
from statsmodels.genmod.generalized_estimating_equations import GEE
from statsmodels.genmod.cov_struct import Exchangeable
from statsmodels.genmod.families import Poisson as GEEPoisson

X_gee = sm.add_constant(df[PREDICTORS])
gee_model = GEE(
    df["claim_count"], X_gee,
    groups=df["Loss Dates"],
    family=GEEPoisson(),
    cov_struct=Exchangeable(),
    offset=offset,
).fit()

print(gee_model.summary())
print(f"\nEstimated within-event correlation (exchangeable): {gee_model.cov_struct.dep_params:.4f}")
print("The estimated correlation is close to zero, but with only two wildfire events this")
print("estimate is highly uncertain and should not be interpreted as evidence of independence.")


                               GEE Regression Results                              
Dep. Variable:                 claim_count   No. Observations:                  232
Model:                                 GEE   No. clusters:                        2
Method:                        Generalized   Min. cluster size:                  80
                      Estimating Equations   Max. cluster size:                 152
Family:                            Poisson   Mean cluster size:               116.0
Dependence structure:         Exchangeable   Num. iterations:                    12
Date:                     Tue, 28 Jul 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         16:28:18
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -0.2138      1.726     -0.124      0.901      -3.59

## 9. Future climate projection: illustrative model output (SSP1-2.6, 2045-2050 vs. 2015-2025)

In [19]:
# Build FSA-level feature tables for both periods, reusing the same
# geospatial feature (dominant_fuel_pct) the model was fit on.
def period_mean_features(daily_fwi_path):
    daily = pd.read_csv(daily_fwi_path)
    return (
        daily.groupby("CFSAUID")[["FWI", "DC"]].mean()
        .rename(columns={"FWI": "fwi_max", "DC": "dc_max"})
        .reset_index().rename(columns={"CFSAUID": "FSA"})
    )

historical_features = period_mean_features(f"{DATA_PROCESSED}/climate/{REGION_NAME}_fsa_fwi_daily.csv")
future_features = period_mean_features(f"{DATA_PROCESSED}/climate/{REGION_NAME}_fsa_fwi_daily_2045_2050.csv")

static_features = pd.read_csv(f"{DATA_PROCESSED}/features/{REGION_NAME}_fsa_static_features.csv")


def build_feature_table(period_features, label):
    merged = period_features.merge(
        static_features[["FSA", "dominant_fuel_pct", "Number of Exposure"]], on="FSA", how="inner"
    )
    n_before = len(merged)
    merged = merged.dropna(subset=PREDICTORS)
    if len(merged) < n_before:
        print(f"{label}: dropped {n_before - len(merged)} FSA(s) with missing predictor data")
    return merged


historical_feature_table = build_feature_table(historical_features, "Historical (2015-2025)")
future_feature_table = build_feature_table(future_features, "Future (2045-2050)")

historical_feature_table[["FSA", "fwi_max", "dc_max", "dominant_fuel_pct"]].head()


,FSA,fwi_max,dc_max,dominant_fuel_pct
0,T0A,1.944436,182.334014,0.235158
1,T0B,2.635500,198.540833,0.751015
2,T0C,2.926940,195.790759,0.586368
3,T0E,1.146894,114.883674,0.209842
4,T0G,1.693942,173.902916,0.298846


In [20]:
# Applies the models fit in sections 3-4 without refitting. Outputs are
# labelled "illustrative" due to a feature-scale mismatch with the training
# data (event-window vs. period-average FWI/DC; see report for detail).
def predict_illustrative(feature_table, label):
    X_period = sm.add_constant(feature_table[PREDICTORS], has_constant="add")
    offset_period = np.log(feature_table["Number of Exposure"])
    feature_table = feature_table.copy()
    feature_table["predicted_count"] = best_freq_model.predict(X_period, offset=offset_period)
    feature_table["predicted_severity"] = severity_model.predict(X_period)
    feature_table["predicted_loss"] = feature_table["predicted_count"] * feature_table["predicted_severity"]

    print(f"{label}: {len(feature_table)} FSAs, mean FWI={feature_table['fwi_max'].mean():.3f}, "
          f"mean DC={feature_table['dc_max'].mean():.2f}, "
          f"illustrative claim frequency={feature_table['predicted_count'].sum():,.1f}, "
          f"illustrative projected loss={feature_table['predicted_loss'].sum():,.0f}")
    return feature_table


historical_pred = predict_illustrative(historical_feature_table, "Historical (2015-2025)")
future_pred = predict_illustrative(future_feature_table, "Future (2045-2050)")


Historical (2015-2025): 154 FSAs, mean FWI=3.469, mean DC=203.56, illustrative claim frequency=1,869.9, illustrative projected loss=4,499,904
Future (2045-2050): 154 FSAs, mean FWI=1.356, mean DC=148.68, illustrative claim frequency=3,861.6, illustrative projected loss=7,654,477


In [21]:
# Step 4: compare historical vs. future. Row labels say "illustrative" for the
# two model-output rows (not "Mean FWI"/"Mean DC", which are just descriptive
# stats of the climate inputs, not model outputs).
summary = pd.DataFrame({
    "Historical baseline (2015-2025)": [
        historical_pred["fwi_max"].mean(),
        historical_pred["dc_max"].mean(),
        historical_pred["predicted_count"].sum(),
        historical_pred["predicted_loss"].sum(),
    ],
    "Illustrative SSP1-2.6 scenario (2045-2050)": [
        future_pred["fwi_max"].mean(),
        future_pred["dc_max"].mean(),
        future_pred["predicted_count"].sum(),
        future_pred["predicted_loss"].sum(),
    ],
}, index=["Mean FWI", "Mean DC", "Illustrative claim frequency", "Illustrative projected loss"])
summary["Relative difference (%)"] = (
    summary["Illustrative SSP1-2.6 scenario (2045-2050)"] / summary["Historical baseline (2015-2025)"] - 1
) * 100
summary


,Historical baseline (2015-2025),Illustrative SSP1-2.6 scenario (2045-2050),Relative difference (%)
Mean FWI,3.469109e+00,1.355943e+00,-60.913791
Mean DC,2.035579e+02,1.486836e+02,-26.957600
Illustrative claim frequency,1.869861e+03,3.861607e+03,106.518436
Illustrative projected loss,4.499904e+06,7.654477e+06,70.103109


In [22]:
fig, ax = plt.subplots(figsize=(5, 4.5))
periods = ["Historical\n(2015-2025)", "Future\n(2045-2050)"]
losses = [historical_pred["predicted_loss"].sum(), future_pred["predicted_loss"].sum()]
bars = ax.bar(periods, losses, color=["#4c72b0", "#c44e52"], width=0.5)
ax.set_ylabel("Illustrative projected loss ($)")
ax.set_title("Illustrative projected portfolio loss: historical vs. future (SSP1-2.6)")
for bar, val in zip(bars, losses):
    ax.text(bar.get_x() + bar.get_width() / 2, val, f"${val / 1e6:.1f}M", ha="center", va="bottom")
ax.set_ylim(0, max(losses) * 1.15)
plt.tight_layout()
plt.savefig(f"../outputs/figures/{REGION_NAME}_expected_loss_historical_vs_future.png", dpi=150, bbox_inches="tight")
plt.show()


Illustrative projected loss rises even though mean FWI falls, because a coefficient learned from only two events can dominate the projection in either direction; this is a property of the fitted coefficients, not evidence about future wildfire risk.

In [23]:
import geopandas as gpd
from matplotlib.colors import TwoSlopeNorm

boundaries = gpd.read_file(f"{DATA_PROCESSED}/boundaries/{REGION_NAME}_fsa_boundaries.gpkg")
boundaries = boundaries.rename(columns={"CFSAUID": "FSA"})

# Percentile ranks improve visual discrimination in the presence of a highly skewed loss distribution.
pooled = pd.concat([
    historical_pred[["FSA", "predicted_loss"]].assign(period="Historical (2015-2025)"),
    future_pred[["FSA", "predicted_loss"]].assign(period="Future (2045-2050)"),
], ignore_index=True)
pooled["loss_pct_rank"] = pooled["predicted_loss"].rank(pct=True) * 100

fig, axes = plt.subplots(1, 3, figsize=(21, 7))

for ax, label in zip(axes[:2], ["Historical (2015-2025)", "Future (2045-2050)"]):
    plot_df = boundaries.merge(pooled.loc[pooled["period"] == label, ["FSA", "loss_pct_rank"]], on="FSA", how="left")
    plot_df.plot(
        column="loss_pct_rank", cmap="OrRd", vmin=0, vmax=100, ax=ax,
        edgecolor="grey", linewidth=0.2, missing_kwds={"color": "lightgrey"},
    )
    ax.set_title(label)
    ax.set_axis_off()

sm_rank = plt.cm.ScalarMappable(cmap="OrRd", norm=plt.Normalize(vmin=0, vmax=100))
sm_rank._A = []
fig.colorbar(sm_rank, ax=axes[:2], shrink=0.6, label="Percentile rank of illustrative projected loss (0=lowest, 100=highest)")

# Third panel: percentage change per FSA, diverging colormap centered at zero.
change_df = historical_pred[["FSA", "predicted_loss"]].merge(
    future_pred[["FSA", "predicted_loss"]], on="FSA", suffixes=("_hist", "_future")
)
change_df["pct_change"] = (change_df["predicted_loss_future"] / change_df["predicted_loss_hist"] - 1) * 100

change_plot_df = boundaries.merge(change_df[["FSA", "pct_change"]], on="FSA", how="left")
vmax_change = change_plot_df["pct_change"].abs().max()
norm = TwoSlopeNorm(vmin=-vmax_change, vcenter=0, vmax=vmax_change)
change_plot_df.plot(
    column="pct_change", cmap="RdBu_r", norm=norm, ax=axes[2],
    edgecolor="grey", linewidth=0.2, missing_kwds={"color": "lightgrey"},
)
sm_change = plt.cm.ScalarMappable(cmap="RdBu_r", norm=norm)
sm_change._A = []
fig.colorbar(sm_change, ax=axes[2], shrink=0.7, label="% change in illustrative projected loss")
axes[2].set_title("% change: future vs. historical")
axes[2].set_axis_off()

plt.suptitle("Illustrative projected portfolio loss by FSA: historical, future, and % change (SSP1-2.6)")
fig.text(
    0.5, 0.02,
    "Percentile ranks are shown rather than dollar values to facilitate visual comparison across periods\n"
    "despite the highly skewed distribution of projected losses.",
    ha="center", fontsize=9, style="italic",
)
plt.savefig(f"../outputs/maps/{REGION_NAME}_expected_loss_by_fsa_choropleth.png", dpi=150, bbox_inches="tight")
plt.show()


The % change panel shows the projected increase is not uniform across the province: it is concentrated in a cluster of central-Alberta FSAs, while some FSAs project a decrease. This spatial pattern follows from local differences in each FSA's climate inputs run through the same two fitted coefficients, not a separate spatial effect.

In [24]:
top_fsas = change_df.sort_values("predicted_loss_future", ascending=False).head(15)["FSA"]
plot_df = change_df.set_index("FSA").loc[top_fsas].sort_values("predicted_loss_future")

fig, ax = plt.subplots(figsize=(8, 6))
y = np.arange(len(plot_df))
bar_h = 0.38
ax.barh(y - bar_h / 2, plot_df["predicted_loss_hist"], height=bar_h, color="#4c72b0", label="Historical (2015-2025)")
ax.barh(y + bar_h / 2, plot_df["predicted_loss_future"], height=bar_h, color="#c44e52", label="Future (2045-2050)")
ax.set_yticks(y)
ax.set_yticklabels(plot_df.index)
ax.set_xlabel("Illustrative projected loss ($)")
ax.set_title("Top 15 FSAs by illustrative future projected loss: historical vs. future")
ax.legend()
plt.tight_layout()
plt.savefig(f"../outputs/figures/{REGION_NAME}_top_fsas_expected_loss.png", dpi=150, bbox_inches="tight")
plt.show()


Ranking FSAs directly avoids the area distortion in the choropleth above and confirms the same top FSAs dominate both periods, with the largest historical-to-future increases concentrated among a handful of them.

## Key Takeaways

- A Negative Binomial frequency model was selected after diagnosing severe overdispersion.
- A parsimonious predictor set combining climate (`fwi_max`, `dc_max`) and a geospatial feature (`dominant_fuel_pct`) provided the baseline model.
- Leave-one-event-out validation demonstrated that two historical wildfire events are insufficient for reliable out-of-sample estimation.
- A GEE sensitivity analysis illustrated a simple treatment of within-event dependence, but additional events would be required for robust inference.
- An illustrative SSP1-2.6 climate scenario showed how the fitted model responds to future climate inputs; these outputs are scenario-based illustrations rather than reliable forecasts.